# AcuDock Pro - Interactive Molecular Docking with CNN Rescoring

**A full-featured docking workbench powered by Gradio.**

AcuDock Pro combines **AutoDock Vina** with **Gnina CNN rescoring** and optional
**Uni-Dock GPU acceleration** in a single interactive interface.

## Features

- **Three scoring modes:** Vina only, Vina + Gnina CNN, or Consensus (z-score weighted)
- **GPU acceleration:** Optional Uni-Dock mode for 1000x+ speedup
- **3D visualization** with interactive 3Dmol.js viewer
- **Multi-pose overlay** to compare binding orientations
- **Batch screening** with ranked results and score chart
- **Lipinski Rule of Five** property checks

## Getting Started

1. **Run the install cell** below (~2-3 min)
2. Runtime will **auto-restart** — this is normal
3. After restart, **skip install cell**, run remaining cells
4. The Gradio app launches inline!

## Why CNN Rescoring?

Vina achieves ~58% redocking success. Gnina's CNN rescoring boosts this to
**~73%** by re-evaluating poses with a neural network trained on PDBBind.

---

**License:** MIT | **Platform:** Google Colab | **Author:** AcuDock Project

In [ ]:
# === Step 1: Install Dependencies ===
# After completion, the runtime restarts. Skip this cell afterward.

!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy gradio matplotlib

# Download Gnina binary for CNN rescoring
!wget -q https://github.com/gnina/gnina/releases/latest/download/gnina -O /content/gnina && chmod +x /content/gnina && echo "Gnina installed" || echo "Gnina download failed"

# Clone AcuDock repo for shared utilities
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || (cd /content/AcuDock && git pull)

# Optional: Install Uni-Dock for GPU-accelerated docking (requires GPU runtime)
# Uncomment the next 2 lines for 1000x+ speedup on NVIDIA GPUs:
# !mkdir -p /content/unidock_env && wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/unidock_env 2>/dev/null && /content/unidock_env/bin/micromamba create -y -p /content/unidock_env/env -c conda-forge unidock && echo "Uni-Dock installed successfully" || echo "Uni-Dock install failed (GPU may not be available)"
# !ls /content/unidock_env/env/bin/unidock 2>/dev/null && echo "Uni-Dock binary found" || echo "Uni-Dock binary not found"

import os
os.kill(os.getpid(), 9)

## Launch AcuDock Pro

Run the cells below to start the interactive interface.

In [ ]:
# === Step 2: Imports ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Add Uni-Dock to PATH if installed via micromamba
if os.path.isdir('/content/unidock_env/env/bin'):
    os.environ['PATH'] = '/content/unidock_env/env/bin:' + os.environ['PATH']

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils

WORK_DIR = '/content/acudock_pro'
os.makedirs(WORK_DIR, exist_ok=True)

# Check engines
print('AcuDock Pro loaded.')
print(utils.get_docking_engine_status())
gnina_ok = os.path.isfile('/content/gnina') and os.access('/content/gnina', os.X_OK)
print(f'Gnina CNN: {"Available" if gnina_ok else "Not installed"}')

In [ ]:
# === Step 3: Launch Gradio Interface ===
import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def run_pro_docking(pdb_id, smiles, lig_name, scoring_mode, engine,
                     exhaustiveness, n_poses, box_size, residues_str,
                     progress=gr.Progress()):
    """Full AcuDock Pro docking pipeline with optional CNN rescoring."""
    log_lines = []
    def log(msg):
        log_lines.append(msg)

    try:
        pdb_id = pdb_id.strip().upper()
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not pdb_id:
            return 'Error: Enter a PDB ID.', None, None, '', '', None, None
        if not smiles or Chem.MolFromSmiles(smiles) is None:
            return 'Error: Invalid SMILES.', None, None, '', '', None, None

        # Protein prep
        progress(0.1, desc='Preparing protein...')
        log(f'Fetching {pdb_id} from PDB...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)
        log('  Protein prepared.')

        # Ligand prep
        progress(0.2, desc='Preparing ligand...')
        log(f'Preparing {lig_name}...')
        ligand_pdbqt, lig_mol = utils.prepare_ligand(smiles, name=lig_name, output_dir=WORK_DIR)
        props = utils.get_ligand_properties(smiles)
        log(f'  MW={props["MW"]} LogP={props["LogP"]} HBD={props["HBD"]} HBA={props["HBA"]}')

        # Lipinski check
        violations = sum([props['MW'] > 500, props['LogP'] > 5,
                          props['HBD'] > 5, props['HBA'] > 10])
        log(f'  Lipinski violations: {violations}/4 ({"PASS" if violations <= 1 else "FAIL"})')

        # Search box
        progress(0.3, desc='Defining search box...')
        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3
        log(f'  Box center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}]')

        # Docking
        progress(0.35, desc='Running docking...')
        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()

        if use_unidock:
            log('Running Uni-Dock GPU docking...')
            energies_raw, poses_path = utils.run_unidock_single(
                receptor_pdbqt, ligand_pdbqt,
                center=center, box_size=box,
                num_modes=int(n_poses), output_dir=WORK_DIR
            )
        else:
            if 'Uni-Dock' in engine:
                log('Uni-Dock unavailable, using Vina.')
            log(f'Running Vina (exhaustiveness={int(exhaustiveness)})...')
            _, energies_arr, poses_path = utils.run_vina(
                receptor_pdbqt, ligand_pdbqt,
                center=center, box_size=box,
                exhaustiveness=int(exhaustiveness), n_poses=int(n_poses)
            )
            energies_raw = [(e[0], e[1], e[2]) for e in energies_arr]

        log(f'  {len(energies_raw)} poses generated.')
        log(f'  Best Vina score: {energies_raw[0][0]:.2f} kcal/mol')

        # CNN Rescoring
        gnina_scores = None
        if scoring_mode in ['Vina + Gnina CNN', 'Consensus']:
            progress(0.65, desc='CNN rescoring...')
            log('Running Gnina CNN rescoring...')
            gnina_raw = utils.run_gnina_rescore(receptor_pdbqt, poses_path, output_dir=WORK_DIR)
            if gnina_raw:
                gnina_scores = [s['value'] for s in gnina_raw if s['metric'] == 'CNNaffinity']
                if gnina_scores:
                    log(f'  CNN rescoring: {len(gnina_scores)} scores')
                else:
                    log('  Warning: No CNN affinity scores parsed.')
                    gnina_scores = None
            else:
                log('  Gnina unavailable, using Vina scores only.')

        # Build results
        progress(0.8, desc='Building results...')
        R, T = 1.987e-3, 298.15

        if scoring_mode == 'Consensus' and gnina_scores and len(gnina_scores) == len(energies_raw):
            consensus_df = utils.consensus_score(
                energies_raw, gnina_scores, alpha=0.5
            )
            results_df = consensus_df
            log('  Consensus ranking computed.')
        else:
            results_df = pd.DataFrame({
                'Pose': range(1, len(energies_raw) + 1),
                'Vina Score': [e[0] for e in energies_raw],
                'RMSD_lb': [round(e[1], 2) for e in energies_raw],
                'RMSD_ub': [round(e[2], 2) for e in energies_raw],
                'Est. Kd (uM)': [round(np.exp(e[0] / (R * T)) * 1e6, 4) for e in energies_raw],
            })
            if gnina_scores and len(gnina_scores) == len(energies_raw):
                results_df['CNN Affinity'] = gnina_scores

        top_score = energies_raw[0][0]
        log(f'\nTop score: {top_score:.2f} kcal/mol ({utils.score_interpretation(top_score)})')

        # 2D image
        mol_2d = Chem.MolFromSmiles(smiles)
        img = Draw.MolToImage(mol_2d, size=(300, 200))
        img_buf = io.BytesIO()
        img.save(img_buf, format='PNG')
        img_buf.seek(0)
        img_path = os.path.join(WORK_DIR, f'{lig_name}_2d.png')
        with open(img_path, 'wb') as f:
            f.write(img_buf.read())

        # 3D viewers
        progress(0.9, desc='Building viewers...')
        with open(protein_pdb, 'r') as f:
            prot_data = f.read()
        with open(poses_path, 'r') as f:
            poses_data = f.read()

        top_pose = utils.extract_pose_from_pdbqt(poses_path, 0)
        single_html = utils.make_3d_viewer_html(prot_data, top_pose)
        multi_html = utils.make_multi_pose_html(prot_data, poses_data, n_poses=3)

        # CSV
        csv_path = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_pro_results.csv')
        results_df.to_csv(csv_path, index=False)

        progress(1.0, desc='Done!')
        return ('\n'.join(log_lines), results_df, img_path,
                single_html, multi_html, csv_path,
                f'{props["MW"]} Da | LogP {props["LogP"]} | HBD {props["HBD"]} | HBA {props["HBA"]} | Lipinski: {"PASS" if violations <= 1 else "FAIL"}')

    except Exception as e:
        log(f'ERROR: {str(e)}')
        return '\n'.join(log_lines), None, None, '', '', None, ''


def run_pro_batch(pdb_id, compounds_text, scoring_mode, engine,
                   exhaustiveness, box_size, residues_str,
                   progress=gr.Progress()):
    """Batch screening with optional CNN rescoring."""
    log_lines = []
    def log(msg):
        log_lines.append(msg)

    try:
        pdb_id = pdb_id.strip().upper()
        if not pdb_id:
            return 'Error: Enter a PDB ID.', None, None, None

        compounds = []
        for line in compounds_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
            if len(parts) == 2:
                name, smi = parts[0].strip(), parts[1].strip()
            else:
                smi = parts[0].strip()
                name = f'Compound_{len(compounds)+1}'
            if Chem.MolFromSmiles(smi) is not None:
                compounds.append((name, smi))

        if not compounds:
            return 'Error: No valid compounds.', None, None, None

        log(f'{len(compounds)} valid compounds.')

        progress(0.05, desc='Preparing protein...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3
        batch_exh = min(int(exhaustiveness), 16)
        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()

        results = []
        for i, (name, smi) in enumerate(compounds):
            pct = 0.1 + 0.8 * (i / len(compounds))
            progress(pct, desc=f'Docking {i+1}/{len(compounds)}: {name}')
            try:
                lig_pdbqt, _ = utils.prepare_ligand(smi, name=name, output_dir=WORK_DIR)
                if use_unidock:
                    en, _ = utils.run_unidock_single(
                        receptor_pdbqt, lig_pdbqt,
                        center=center, box_size=box,
                        num_modes=5, output_dir=WORK_DIR
                    )
                    best = en[0][0] if en else None
                else:
                    _, en, _ = utils.run_vina(
                        receptor_pdbqt, lig_pdbqt,
                        center=center, box_size=box,
                        exhaustiveness=batch_exh, n_poses=5
                    )
                    best = en[0][0] if len(en) > 0 else None

                mp = utils.get_ligand_properties(smi)
                R, T = 1.987e-3, 298.15
                results.append({
                    'Name': name, 'SMILES': smi,
                    'Score': round(best, 2) if best else None,
                    'Est. Kd (uM)': round(np.exp(best / (R * T)) * 1e6, 4) if best else None,
                    'MW': mp.get('MW'), 'LogP': mp.get('LogP'),
                    'Interpretation': utils.score_interpretation(best) if best else 'Failed',
                })
                log(f'  [{i+1}] {name}: {best:.2f} kcal/mol')
            except Exception as e:
                log(f'  [{i+1}] {name}: FAILED')
                results.append({'Name': name, 'SMILES': smi, 'Score': None,
                                'Est. Kd (uM)': None, 'MW': None, 'LogP': None,
                                'Interpretation': 'Failed'})

        batch_df = pd.DataFrame(results).sort_values('Score', ascending=True).reset_index(drop=True)
        batch_df.insert(0, 'Rank', range(1, len(batch_df) + 1))

        # Chart
        valid = batch_df.dropna(subset=['Score'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.4)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Score']]
        ax.barh(valid['Name'], valid['Score'], color=colors, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'Batch Screening: {pdb_id}')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5)
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'batch_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        csv_path = os.path.join(WORK_DIR, f'batch_{pdb_id}_pro.csv')
        batch_df.to_csv(csv_path, index=False)

        progress(1.0, desc='Done!')
        return '\n'.join(log_lines), batch_df, chart_path, csv_path

    except Exception as e:
        log(f'ERROR: {str(e)}')
        return '\n'.join(log_lines), None, None, None


# ---------------------------------------------------------------------------
# Gradio Interface
# ---------------------------------------------------------------------------

DEFAULT_COMPOUNDS = """Aspirin, CC(=O)Oc1ccccc1C(=O)O
Ibuprofen, CC(C)Cc1ccc(cc1)C(C)C(=O)O
Caffeine, Cn1c(=O)c2c(ncn2C)n(C)c1=O
Acetaminophen, CC(=O)Nc1ccc(O)cc1
Naproxen, COc1ccc2cc(ccc2c1)C(C)C(=O)O
Metformin, CN(C)C(=N)NC(=N)N
Celecoxib, Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1
Diclofenac, OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl
Atorvastatin, CC(C)c1n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c(c2ccc(F)cc2)c(c1c1ccccc1)C(=O)Nc1ccccc1
Omeprazole, COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C"""

with gr.Blocks(
    title='AcuDock Pro',
    theme=gr.themes.Soft(),
    css='.gradio-container { max-width: 1200px !important; }'
) as demo:

    gr.Markdown("""
    # AcuDock Pro
    **Interactive molecular docking with CNN rescoring and GPU acceleration.**
    """)

    with gr.Tabs():

        # === Single Docking Tab ===
        with gr.Tab('Single Docking'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Target')
                    pdb_in = gr.Textbox(label='PDB ID', value='1HSG')
                    residues_in = gr.Textbox(
                        label='Active Site Residues',
                        value='23,24,25,26,27,28,29,30'
                    )

                    gr.Markdown('### Ligand')
                    smiles_in = gr.Textbox(
                        label='SMILES',
                        value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
                        lines=2
                    )
                    name_in = gr.Textbox(label='Name', value='Indinavir')

                    gr.Markdown('### Scoring & Engine')
                    scoring_in = gr.Radio(
                        ['Vina', 'Vina + Gnina CNN', 'Consensus'],
                        value='Vina', label='Scoring Mode'
                    )
                    engine_in = gr.Radio(
                        ['Vina (CPU)', 'Uni-Dock (GPU)'],
                        value='Vina (CPU)', label='Docking Engine'
                    )

                    gr.Markdown('### Parameters')
                    exh_in = gr.Slider(8, 128, value=32, step=8, label='Exhaustiveness')
                    poses_in = gr.Slider(5, 50, value=20, step=5, label='Max Poses')
                    box_in = gr.Slider(15, 40, value=20, step=5, label='Box Size (A)')

                    dock_btn = gr.Button('Run Docking', variant='primary', size='lg')

                with gr.Column(scale=2):
                    log_out = gr.Textbox(label='Log', lines=8, interactive=False)
                    props_out = gr.Textbox(label='Molecular Properties', interactive=False)
                    with gr.Row():
                        img_out = gr.Image(label='2D Structure', height=200)
                    results_out = gr.Dataframe(label='Results')

                    with gr.Tabs():
                        with gr.Tab('Top Pose'):
                            viewer_single = gr.HTML(label='3D Viewer')
                        with gr.Tab('Multi-Pose Overlay'):
                            viewer_multi = gr.HTML(label='Pose Overlay')

                    csv_out = gr.File(label='Download CSV')

            dock_btn.click(
                fn=run_pro_docking,
                inputs=[pdb_in, smiles_in, name_in, scoring_in, engine_in,
                        exh_in, poses_in, box_in, residues_in],
                outputs=[log_out, results_out, img_out,
                         viewer_single, viewer_multi, csv_out, props_out]
            )

        # === Batch Screening Tab ===
        with gr.Tab('Batch Screening'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Batch Configuration')
                    b_pdb = gr.Textbox(label='PDB ID', value='1HSG')
                    b_residues = gr.Textbox(label='Active Site', value='23,24,25,26,27,28,29,30')
                    b_scoring = gr.Radio(['Vina', 'Vina + Gnina CNN'], value='Vina', label='Scoring')
                    b_engine = gr.Radio(['Vina (CPU)', 'Uni-Dock (GPU)'], value='Vina (CPU)', label='Engine')
                    b_exh = gr.Slider(8, 64, value=8, step=8, label='Exhaustiveness')
                    b_box = gr.Slider(15, 40, value=20, step=5, label='Box Size (A)')
                    b_compounds = gr.Textbox(
                        label='Compounds (Name, SMILES per line)',
                        value=DEFAULT_COMPOUNDS, lines=10
                    )
                    b_btn = gr.Button('Run Batch', variant='primary', size='lg')

                with gr.Column(scale=2):
                    b_log = gr.Textbox(label='Log', lines=8, interactive=False)
                    b_table = gr.Dataframe(label='Batch Results')
                    b_chart = gr.Image(label='Score Chart')
                    b_csv = gr.File(label='Download CSV')

            b_btn.click(
                fn=run_pro_batch,
                inputs=[b_pdb, b_compounds, b_scoring, b_engine, b_exh, b_box, b_residues],
                outputs=[b_log, b_table, b_chart, b_csv]
            )

        # === About Tab ===
        with gr.Tab('About'):
            gr.Markdown("""
            ### AcuDock Pro

            **Pipeline:**
            ```
            PDB ID --> PDBFixer --> PDBQT
            SMILES --> RDKit 3D --> Meeko --> PDBQT
                     |                        |
                     v                        v
                 Vina / Uni-Dock --> Gnina CNN Rescore --> Consensus
                     |                                        |
                     +---------> 3D Visualization <-----------+
            ```

            **Scoring Modes:**
            | Mode | Description | Accuracy |
            |---|---|---|
            | Vina | Classical empirical | ~58% redocking |
            | Vina + Gnina CNN | CNN-based rescoring | ~73% redocking |
            | Consensus | Z-score weighted | Best overall |

            **Engines:**
            - **Vina (CPU):** Reliable, no GPU needed
            - **Uni-Dock (GPU):** 1000x+ speedup on NVIDIA GPUs

            *MIT License | AcuDock Project*
            """)

demo.launch()